# 06: Results

Comparison tables, Diebold-Mariano tests, and final figures across both regimes.

All numbers come from saved CSVs produced in notebooks 02-05. No retraining here.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import joblib
from pathlib import Path
from scipy import stats
from sklearn.metrics import mean_squared_error, r2_score

DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../outputs")
MODEL_DIR = OUTPUT_DIR / "models"
FIGURES_DIR = OUTPUT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## Load saved results

In [ ]:
calm_benchmarks = pd.read_csv(OUTPUT_DIR / "benchmark_results.csv")
calm_mlp = pd.read_csv(OUTPUT_DIR / "mlp_results.csv")
stress_benchmarks = pd.read_csv(OUTPUT_DIR / "benchmark_results_stress.csv")
stress_mlp = pd.read_csv(OUTPUT_DIR / "mlp_results_stress.csv")

calm_results = pd.concat([calm_benchmarks, calm_mlp], ignore_index=True)
stress_results = pd.concat([stress_benchmarks, stress_mlp], ignore_index=True)

results = calm_results.merge(stress_results, on="model", suffixes=("_calm", "_stress"))
print(results[["model", "mse_calm", "r2_calm", "mse_stress", "r2_stress"]].to_string(index=False))

## Full results table

In [ ]:
display_cols = ["model", "mse_calm", "r2_calm", "mse_stress", "r2_stress"]
display = results[display_cols].copy()
display.columns = ["Model", "Calm MSE", "Calm R2", "Stress MSE", "Stress R2"]

for col in ["Calm MSE", "Calm R2", "Stress MSE", "Stress R2"]:
    display[col] = display[col].round(4)

display.to_csv(OUTPUT_DIR / "results_combined.csv", index=False)
print(display.to_string(index=False))

## Diebold-Mariano tests

Tests whether MLP-A and MLP-B significantly outperform the Almgren-Chriss benchmark
on each regime's test set. Two-sided, H0: equal predictive accuracy.

In [ ]:
def diebold_mariano(e1: np.ndarray, e2: np.ndarray, h: int = 1) -> tuple[float, float]:
    """
    Harvey, Leybourne & Newbold (1997) small-sample corrected DM test.
    e1, e2: forecast errors (not squared). Returns (dm_stat, p_value).
    """
    d = e1**2 - e2**2
    n = len(d)
    d_bar = d.mean()
    gamma = np.array([np.cov(d[h:], d[:-h])[0, 1] if h > 0 else 0
                      for h in range(h + 1)])
    var_d = (gamma[0] + 2 * gamma[1:].sum()) / n
    dm = d_bar / np.sqrt(var_d)
    p = 2 * (1 - stats.norm.cdf(np.abs(dm)))
    return float(dm), float(p)


class ImpactMLP(nn.Module):
    def __init__(self, input_dim: int, hidden: int = 64, layers: int = 3, dropout: float = 0.3):
        super().__init__()
        net = [nn.Linear(input_dim, hidden), nn.ReLU(), nn.Dropout(dropout)]
        for _ in range(layers - 1):
            net += [nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout)]
        net += [nn.Linear(hidden, 1)]
        self.net = nn.Sequential(*net)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


def predict(model: ImpactMLP, X: np.ndarray) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32).to(DEVICE)).cpu().numpy()


def prepare(df: pd.DataFrame) -> pd.DataFrame:
    df = df[(df["I"] > 0) & (df["Q_norm"] > 0) & (df["sigma_daily"] > 0)].copy()
    df["log_I"] = np.log(df["I"])
    df["log_Q"] = np.log(df["Q_norm"])
    df["log_sigma"] = np.log(df["sigma_daily"])
    df["log_V"] = np.log(df["V_daily"])
    df["log_n_child"] = np.log(df["n_child"])
    hour = pd.to_datetime(df["start_ts"], unit="ms").dt.hour
    df["utc_hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["utc_hour_cos"] = np.cos(2 * np.pi * hour / 24)
    mu, sd = df["log_I"].mean(), df["log_I"].std()
    return df[np.abs(df["log_I"] - mu) < 3 * sd].reset_index(drop=True)

In [ ]:
FEATURES_A = ["log_Q", "log_sigma", "log_V"]
FEATURES_B = ["log_Q", "log_sigma", "log_V", "log_n_child", "utc_hour_sin", "utc_hour_cos"]

from sklearn.linear_model import LinearRegression

dm_results = []

for regime, test_path, mlp_a_path, mlp_b_path, scaler_a_path, scaler_b_path in [
    ("calm",   DATA_DIR / "mo_test.parquet",        MODEL_DIR / "mlp_a.pt",        MODEL_DIR / "mlp_b.pt",        MODEL_DIR / "scaler_a.pkl",        MODEL_DIR / "scaler_b.pkl"),
    ("stress", DATA_DIR / "mo_test_stress.parquet", MODEL_DIR / "mlp_a_stress.pt", MODEL_DIR / "mlp_b_stress.pt", MODEL_DIR / "scaler_a_stress.pkl", MODEL_DIR / "scaler_b_stress.pkl"),
]:
    test_p = prepare(pd.read_parquet(test_path))
    y_te = test_p["log_I"].values

    # Almgren-Chriss benchmark
    X_b2 = test_p[["log_Q", "log_sigma"]].values
    train_p = prepare(pd.read_parquet(test_path.parent / test_path.name.replace("test", "train")))
    b2 = LinearRegression().fit(train_p[["log_Q", "log_sigma"]].values, train_p["log_I"].values)
    e_b2 = y_te - b2.predict(X_b2)

    scaler_a = joblib.load(scaler_a_path)
    scaler_b = joblib.load(scaler_b_path)

    mlp_a = ImpactMLP(input_dim=len(FEATURES_A)).to(DEVICE)
    mlp_a.load_state_dict(torch.load(mlp_a_path, map_location=DEVICE))
    mlp_b = ImpactMLP(input_dim=len(FEATURES_B)).to(DEVICE)
    mlp_b.load_state_dict(torch.load(mlp_b_path, map_location=DEVICE))

    e_a = y_te - predict(mlp_a, scaler_a.transform(test_p[FEATURES_A].values))
    e_b = y_te - predict(mlp_b, scaler_b.transform(test_p[FEATURES_B].values))

    dm_a, p_a = diebold_mariano(e_b2, e_a)
    dm_b, p_b = diebold_mariano(e_b2, e_b)

    dm_results.append({"regime": regime, "comparison": "AC vs MLP-A", "DM": dm_a, "p": p_a})
    dm_results.append({"regime": regime, "comparison": "AC vs MLP-B", "DM": dm_b, "p": p_b})
    print(f"{regime}  AC vs MLP-A: DM={dm_a:.3f}  p={p_a:.4f}")
    print(f"{regime}  AC vs MLP-B: DM={dm_b:.3f}  p={p_b:.4f}")

dm_df = pd.DataFrame(dm_results)
dm_df.to_csv(OUTPUT_DIR / "dm_tests.csv", index=False)

## PySR formula comparison table

In [ ]:
calm_summary = pd.read_json(OUTPUT_DIR / "calm_summary.json", typ="series")
stress_summary = pd.read_json(OUTPUT_DIR / "stress_summary.json", typ="series")

def log_q_in_formula(eq: str) -> str:
    return "Yes (negative)" if "log_Q" in eq else "No"

pysr_comparison = pd.DataFrame([
    {
        "Regime": "Calm",
        "OLS delta": round(calm_summary["ols_delta"], 3),
        "MLP mean slope": round(calm_summary["mlp_mean_slope"], 3),
        "PySR formula": calm_summary["pysr_equation"],
        "log_Q in formula": log_q_in_formula(calm_summary["pysr_equation"]),
        "Matches equity model": "No",
    },
    {
        "Regime": "Stress",
        "OLS delta": round(stress_summary["ols_delta"], 3),
        "MLP mean slope": round(stress_summary["mlp_mean_slope"], 3),
        "PySR formula": stress_summary["pysr_equation"],
        "log_Q in formula": log_q_in_formula(stress_summary["pysr_equation"]),
        "Matches equity model": "No",
    },
])

pysr_comparison.to_csv(OUTPUT_DIR / "pysr_comparison.csv", index=False)
print(pysr_comparison.to_string(index=False))

## Figure: OOS MSE by model and regime

In [ ]:
labels = [
    "Power law\n(delta=0.5)",
    "Power law\n(fitted)",
    "Almgren-Chriss",
    "MLP-A",
    "MLP-B",
]

calm_mse = results["mse_calm"].values
stress_mse = results["mse_stress"].values

x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(x - w/2, calm_mse, width=w, color="steelblue", label="Calm")
ax.bar(x + w/2, stress_mse, width=w, color="firebrick", label="Stress")

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("OOS MSE")
ax.set_title("Out-of-sample MSE by model and regime")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "oos_mse_comparison.png", bbox_inches="tight")
plt.show()

## Figure: Delta across regimes and methods

In [ ]:
calm_summary = pd.read_json(OUTPUT_DIR / "calm_summary.json", typ="series")
stress_summary = pd.read_json(OUTPUT_DIR / "stress_summary.json", typ="series")

delta_data = {
    "Method": ["OLS power law", "MLP-A mean slope", "Sqrt law (theory)"],
    "Calm":   [calm_summary["ols_delta"], calm_summary["mlp_mean_slope"], 0.5],
    "Stress": [stress_summary["ols_delta"], stress_summary["mlp_mean_slope"], 0.5],
}

delta_df = pd.DataFrame(delta_data)

x = np.arange(len(delta_df))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(x - w/2, delta_df["Calm"], width=w, color="steelblue", label="Calm")
ax.bar(x + w/2, delta_df["Stress"], width=w, color="firebrick", label="Stress")

ax.set_xticks(x)
ax.set_xticklabels(delta_df["Method"])
ax.set_ylabel("delta")
ax.set_title("Size exponent (delta) by method and regime")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="Sqrt law (0.5)")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "delta_comparison.png", bbox_inches="tight")
plt.show()


## Figure: Slope comparison calm vs stress (normalized)

In [ ]:
def prepare_calm(df: pd.DataFrame) -> pd.DataFrame:
    df = df[(df["I"] > 0) & (df["Q_norm"] > 0) & (df["sigma_daily"] > 0)].copy()
    df["log_I"] = np.log(df["I"])
    df["log_Q"] = np.log(df["Q_norm"])
    df["log_sigma"] = np.log(df["sigma_daily"])
    df["log_V"] = np.log(df["V_daily"])
    mu, sd = df["log_I"].mean(), df["log_I"].std()
    return df[np.abs(df["log_I"] - mu) < 3 * sd].reset_index(drop=True)

mo_train_calm = prepare_calm(pd.read_parquet(DATA_DIR / "mo_train.parquet"))
mo_train_stress = prepare(pd.read_parquet(DATA_DIR / "mo_train_stress.parquet"))

scaler_a_calm = joblib.load(MODEL_DIR / "scaler_a.pkl")
scaler_a_stress = joblib.load(MODEL_DIR / "scaler_a_stress.pkl")

mlp_a_calm = ImpactMLP(input_dim=3).to(DEVICE)
mlp_a_calm.load_state_dict(torch.load(MODEL_DIR / "mlp_a.pt", map_location=DEVICE))

mlp_a_stress = ImpactMLP(input_dim=3).to(DEVICE)
mlp_a_stress.load_state_dict(torch.load(MODEL_DIR / "mlp_a_stress.pt", map_location=DEVICE))

median_sigma_calm = mo_train_calm["log_sigma"].median()
median_V_calm = mo_train_calm["log_V"].median()
median_sigma_stress = mo_train_stress["log_sigma"].median()
median_V_stress = mo_train_stress["log_V"].median()

q_range = np.linspace(
    mo_train_stress["log_Q"].quantile(0.01),
    mo_train_stress["log_Q"].quantile(0.99),
    200,
)

X_calm_raw = np.column_stack([
    q_range,
    np.full_like(q_range, median_sigma_calm),
    np.full_like(q_range, median_V_calm),
])
X_stress_raw = np.column_stack([
    q_range,
    np.full_like(q_range, median_sigma_stress),
    np.full_like(q_range, median_V_stress),
])

y_calm_mlp = predict(mlp_a_calm, scaler_a_calm.transform(X_calm_raw))
y_stress_mlp = predict(mlp_a_stress, scaler_a_stress.transform(X_stress_raw))

# Load PySR models
model_pysr_calm = joblib.load(MODEL_DIR / "pysr_calm.pkl")
calm_elbow_idx = joblib.load(MODEL_DIR / "pysr_calm_elbow_idx.pkl")
y_calm_pysr = model_pysr_calm.predict(X_calm_raw, index=calm_elbow_idx)

model_pysr_stress = joblib.load(MODEL_DIR / "pysr_stress.pkl")
stress_elbow_idx = joblib.load(MODEL_DIR / "pysr_stress_elbow_idx.pkl")
y_stress_pysr = model_pysr_stress.predict(X_stress_raw, index=stress_elbow_idx)

mid = len(q_range) // 2
y_sqrt_line = 0.5 * (q_range - q_range[mid])

y_calm_mlp_norm = y_calm_mlp - y_calm_mlp[0]
y_calm_pysr_norm = y_calm_pysr - y_calm_pysr[0]
y_stress_mlp_norm = y_stress_mlp - y_stress_mlp[0]
y_stress_pysr_norm = y_stress_pysr - y_stress_pysr[0]
y_sqrt_norm = y_sqrt_line - y_sqrt_line[0]

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(q_range, y_calm_mlp_norm, color="steelblue", linewidth=2, linestyle="--", label="MLP-A (calm)")
ax.plot(q_range, y_calm_pysr_norm, color="steelblue", linewidth=1.5, linestyle=":", label="PySR (calm)")
ax.plot(q_range, y_stress_mlp_norm, color="firebrick", linewidth=2, linestyle="--", label="MLP-A (stress)")
ax.plot(q_range, y_stress_pysr_norm, color="firebrick", linewidth=1.5, linestyle=":", label="PySR (stress)")
ax.plot(q_range, y_sqrt_norm, color="gray", linewidth=1.5, linestyle="-", label="Sqrt law (delta=0.5)")

ax.set_xlabel("log(Q_norm)")
ax.set_ylabel("Predicted log I (normalized to zero)")
ax.set_title("Impact formula slope comparison: calm vs stress")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "formula_comparison_regimes_06.png", bbox_inches="tight")
plt.show()

## Summary

In [ ]:
calm_summary = pd.read_json(OUTPUT_DIR / "calm_summary.json", typ="series")
stress_summary = pd.read_json(OUTPUT_DIR / "stress_summary.json", typ="series")

best_calm_r2 = results["r2_calm"].max()
best_calm_model = results.loc[results["r2_calm"].idxmax(), "model"]
best_stress_r2 = results["r2_stress"].max()
best_stress_model = results.loc[results["r2_stress"].idxmax(), "model"]

calm_log_q = "Yes" if "log_Q" in calm_summary["pysr_equation"] else "No"
stress_log_q = "Yes" if "log_Q" in stress_summary["pysr_equation"] else "No"

print("Key results:")
print()
print(f"  {'':35} {'Calm':>10} {'Stress':>10}")
print(f"  {'-'*57}")
print(f"  {'OLS delta':<35} {calm_summary['ols_delta']:>10.3f} {stress_summary['ols_delta']:>10.3f}")
print(f"  {'MLP-A mean local slope':<35} {calm_summary['mlp_mean_slope']:>10.3f} {stress_summary['mlp_mean_slope']:>10.3f}")
print(f"  {'Sqrt law (theory)':<35} {0.5:>10.3f} {0.5:>10.3f}")
print(f"  {'Best OOS R2':<35} {f'{best_calm_r2:.3f} ({best_calm_model})':>10} {f'{best_stress_r2:.3f} ({best_stress_model})':>10}")
print(f"  {'log_Q in PySR formula':<35} {calm_log_q:>10} {stress_log_q:>10}")
print(f"  {'Matches equity model':<35} {'No':>10} {'No':>10}")
print()
print("DM test results:")
print(dm_df.to_string(index=False))